[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-02-character-ngram-language-model.ipynb)

# Advanced Discussion 2 — a language model with no neural network

Character n-gram models, smoothing and back-off, perplexity, and sampling at different temperatures.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Downloads Tiny Shakespeare (1 MB); falls back to a short built-in passage if the download fails.

In [ ]:
import math, urllib.request, numpy as np
from collections import Counter, defaultdict
np.random.seed(0)

## 1. text: Tiny Shakespeare (1.1 MB); offline fallback = a short built-in passage

In [ ]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
try: text = urllib.request.urlopen(URL, timeout=20).read().decode("utf-8")
except Exception: text = ("To be, or not to be, that is the question: Whether 'tis nobler in the mind to suffer the slings and arrows of outrageous fortune, "
                        "or to take arms against a sea of troubles and by opposing end them. ") * 200
text = text[:600_000]
cut = int(0.9 * len(text)); train, heldout = text[:cut], text[cut:]
vocab = sorted(set(text)); V = len(vocab); print("characters", len(text), "vocabulary", V)

## 2. character n-gram models: counts, then three ways of turning counts into probabilities

In [ ]:
def build(order):
    counts = defaultdict(Counter)
    for i in range(order - 1, len(train)):
        counts[train[i - order + 1:i]][train[i]] += 1
    return counts
models = {n: build(n) for n in range(1, 8)}
def p_mle(n, ctx, ch):
    c = models[n].get(ctx); return (c[ch] / sum(c.values())) if c and ch in c else 0.0
def p_addk(n, ctx, ch, k=0.05):
    c = models[n].get(ctx, Counter()); return (c[ch] + k) / (sum(c.values()) + k * V)
def p_interp(n, ctx, ch, lam=0.75):                       # interpolate with the next-shorter context, recursively
    if n == 1: return p_addk(1, "", ch, 0.5)
    c = models[n].get(ctx); lower = p_interp(n - 1, ctx[1:], ch, lam)
    if not c: return lower
    tot = sum(c.values()); distinct = len(c)
    disc = 0.75                                            # absolute discounting: take a fixed amount from each seen event, give it to the lower order
    return max(c[ch] - disc, 0) / tot + (disc * distinct / tot) * lower
def perplexity(prob, n, sample=20000):
    ll, cnt = 0.0, 0
    for i in range(n - 1, min(len(heldout), sample + n)):
        p = prob(n, heldout[i - n + 1:i], heldout[i]); ll += math.log(max(p, 1e-12)); cnt += 1
    return math.exp(-ll / cnt)
print("order  MLE      add-k    interpolated absolute discounting")
for n in range(1, 8):
    print("%5d  %7.2f  %7.2f  %7.2f" % (n, perplexity(p_mle, n), perplexity(p_addk, n), perplexity(p_interp, n)))
print("uniform baseline perplexity =", V)

## 3. sampling from the best model: temperature changes what comes out

In [ ]:
def sample_text(n, length, temperature, seed):
    rng = np.random.RandomState(seed); out = "First Citizen:\n"
    for _ in range(length):
        ctx = out[-(n - 1):]; probs = np.array([p_interp(n, ctx, ch) for ch in vocab]); probs = probs ** (1 / temperature); probs /= probs.sum()
        out += vocab[rng.choice(V, p=probs)]
    return out
for T in (0.5, 1.0, 1.5):
    s = sample_text(5, 200, T, 0)[len("First Citizen:\n"):]
    print("--- temperature", T); print(s[:160].replace("\n", " / "))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.